# What Matters in a Small Language Model — Colab runner

Runs the AFML default project: architecture ablations, scaling sweep, instruction tuning and evaluation.
**The whole team can run this notebook at the same time.** Everyone shares one Drive folder; each session
claims the next job nobody else is running, so work splits itself.

**One-time setup per teammate**
1. Dhyan shares the Drive folder `AFML-SLM` with you as **Editor**.
2. In Google Drive, open *Shared with me*, right-click `AFML-SLM` → *Organize → Add shortcut* → **My Drive**.
   The shortcut must sit directly in My Drive and be named exactly `AFML-SLM`.

**Every session**
1. *Runtime → Change runtime type →* **T4 GPU**.
2. Type your name in the first code cell below.
3. *Runtime → Run all*, and allow Drive access **with the same Google account you added the shortcut in**.
4. Keep the tab open. If Colab disconnects or runs out of time, just **Run all** again later. Finished jobs are
   skipped; a job whose session died is picked up again after 15 minutes (by you or anyone) and resumes from its
   last checkpoint.

Results, logs and checkpoints go to `AFML-SLM/out`; prepared data is cached in `AFML-SLM/data_cache`.

## Who is running this session?

In [ ]:
WORKER = ''  # <- your first name, e.g. 'dhruv'. Every teammate uses a different name.

# Order to work through the steps. Step 2's largest model is the longest job (~7 h of GPU time across
# sessions), so at least one person should start with Step 2: STEPS = [2, 1, 3, 4]
STEPS = [1, 2, 3, 4]

WORKER = WORKER.strip().lower().replace(' ', '_')
assert WORKER, 'Type your name in WORKER above, then Run all again.'

## Setup

In [ ]:
import os, shutil, time
from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/AFML-SLM'  # shared results and cached data
WORK = '/content/slm'                     # code and training data (fast local disk)
if not os.path.isdir(DRIVE):
    raise SystemExit('MyDrive/AFML-SLM not found. Add the shared AFML-SLM folder to My Drive as a shortcut '
                     '(see the instructions at the top), and mount Drive with that same Google account.')
for d in (f'{DRIVE}/out', f'{DRIVE}/data_cache'):
    os.makedirs(d, exist_ok=True)
for d in ('slm', 'scripts', 'configs', 'tests', 'data'):
    os.makedirs(f'{WORK}/{d}', exist_ok=True)
if not os.path.islink(f'{WORK}/out'):
    os.symlink(f'{DRIVE}/out', f'{WORK}/out')
os.chdir(WORK)

!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q tokenizers datasets

## Project code
Generated from the repository by `scripts/make_colab_notebook.py`; edit the repository, not these cells.

In [ ]:
%%writefile slm/__init__.py
# (empty)


In [ ]:
%%writefile slm/chat.py
"""Chat template and supervised fine-tuning examples.

One example is laid out as

    <|endoftext|><|user|>{prompt}<|end|><|assistant|>{response}<|end|>

With mask_prompt=True the loss covers only the response and its closing <|end|>,
so the model learns to answer rather than to imitate user prompts.
"""

import torch

from .tokenizer import ASSISTANT_ID, END_ID, EOT_ID, USER_ID

IGNORE = -100


def prompt_ids(tok, prompt: str) -> list[int]:
    """Tokens the model sees before it starts answering."""
    return [EOT_ID, USER_ID] + tok.encode(prompt).ids + [END_ID, ASSISTANT_ID]


def encode_example(tok, prompt: str, response: str, block_size: int, mask_prompt: bool = True):
    """Return (inputs, targets) as int lists, or None if the example is too long."""
    p = prompt_ids(tok, prompt)
    ids = p + tok.encode(response).ids + [END_ID]
    if len(ids) > block_size + 1:
        return None
    x, y = ids[:-1], ids[1:]
    if mask_prompt:
        # y[j] is the token after x[j]; the first response token is y[len(p) - 1].
        y = [IGNORE] * (len(p) - 1) + y[len(p) - 1 :]
    return x, y


def collate(examples, device: str):
    """Right-pad a list of (x, y) pairs into tensors. Padded targets are ignored."""
    T = max(len(x) for x, _ in examples)
    xs = torch.full((len(examples), T), EOT_ID, dtype=torch.long)
    ys = torch.full((len(examples), T), IGNORE, dtype=torch.long)
    for i, (x, y) in enumerate(examples):
        xs[i, : len(x)] = torch.tensor(x)
        ys[i, : len(y)] = torch.tensor(y)
    return xs.to(device), ys.to(device)


In [ ]:
%%writefile slm/data.py
"""Pretraining batches sampled from a flat binary file of uint16 token ids."""

import os

import numpy as np
import torch


class TokenBins:
    def __init__(self, data_dir: str, block_size: int, device: str):
        self.data_dir = data_dir
        self.block_size = block_size
        self.device = device

    def _tokens(self, split: str):
        # Re-open the memmap per call: holding one open across a long run leaks memory.
        return np.memmap(os.path.join(self.data_dir, f"{split}.bin"), dtype=np.uint16, mode="r")

    def num_tokens(self, split: str) -> int:
        return len(self._tokens(split))

    def _batch_at(self, data, offsets):
        T = self.block_size
        x = torch.from_numpy(np.stack([data[i : i + T].astype(np.int64) for i in offsets]))
        y = torch.from_numpy(np.stack([data[i + 1 : i + 1 + T].astype(np.int64) for i in offsets]))
        return x, y

    def _to_device(self, x, y):
        if self.device.startswith("cuda"):
            return x.pin_memory().to(self.device, non_blocking=True), y.pin_memory().to(self.device, non_blocking=True)
        return x.to(self.device), y.to(self.device)

    def get_batch(self, split: str, batch_size: int):
        data = self._tokens(split)
        offsets = torch.randint(len(data) - self.block_size - 1, (batch_size,)).tolist()
        return self._to_device(*self._batch_at(data, offsets))

    def fixed_batches(self, split: str, n_batches: int, batch_size: int, seed: int = 1234):
        """The same evaluation batches every call and every run, so validation
        losses from different ablation runs are measured on identical tokens."""
        data = self._tokens(split)
        g = torch.Generator().manual_seed(seed)
        batches = []
        for _ in range(n_batches):
            offsets = torch.randint(len(data) - self.block_size - 1, (batch_size,), generator=g).tolist()
            batches.append(self._batch_at(data, offsets))
        return batches


In [ ]:
%%writefile slm/lora.py
"""Low-rank adaptation (Hu et al., 2021) written out by hand.

A frozen weight W is adapted as W + (alpha / r) * B @ A, where A is (r x in) and
B is (out x r). B starts at zero, so the adapted model is exactly the base model
at step 0 and training only has to learn the low-rank update.
"""

import math

import torch
import torch.nn as nn

DEFAULT_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj")


class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float = 0.0):
        super().__init__()
        self.base = base
        self.base.weight.requires_grad_(False)
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.scaling = alpha / r
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        update = (self.dropout(x) @ self.lora_A.T) @ self.lora_B.T
        return self.base(x) + update * self.scaling

    def merged(self) -> nn.Linear:
        """Fold the update into a plain Linear: no extra cost at inference time."""
        out = nn.Linear(self.base.in_features, self.base.out_features, bias=False)
        delta = (self.lora_B @ self.lora_A) * self.scaling
        out.weight.data.copy_(self.base.weight.data + delta.to(self.base.weight.dtype))
        return out.to(self.base.weight.device)


def apply_lora(model: nn.Module, r: int, alpha: float, dropout: float = 0.0, targets=DEFAULT_TARGETS):
    """Freeze every base weight, then wrap each Linear whose attribute name is in targets."""
    for p in model.parameters():
        p.requires_grad_(False)
    for module in list(model.modules()):
        for name, child in list(module.named_children()):
            if name in targets and isinstance(child, nn.Linear):
                setattr(module, name, LoRALinear(child, r, alpha, dropout))
    return model


def merge_lora(model: nn.Module):
    """Replace every LoRALinear with its merged Linear, in place."""
    for module in list(model.modules()):
        for name, child in list(module.named_children()):
            if isinstance(child, LoRALinear):
                setattr(module, name, child.merged())
    for p in model.parameters():
        p.requires_grad_(True)
    return model


def lora_state_dict(model: nn.Module) -> dict:
    return {k: v for k, v in model.state_dict().items() if "lora_" in k}


In [ ]:
%%writefile slm/model.py
"""Decoder-only transformer with swappable components, built for ablations.

Each architecture choice the project tests is a config switch, so a GPT-2-style
baseline and a LLaMA-style model are the same class with different settings:

    pos_emb    "learned" (GPT-2)    | "rope" (LLaMA)
    norm       "layernorm" (GPT-2)  | "rmsnorm" (LLaMA)
    mlp        "gelu" (GPT-2)       | "swiglu" (LLaMA)
    n_kv_head  == n_head (multi-head attention) | < n_head (grouped-query attention)

All variants use pre-norm residual blocks, bias-free linear layers and tied
input/output embeddings, so only the switched component differs between runs.
"""

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    vocab_size: int = 8192
    block_size: int = 512
    n_layer: int = 6
    n_head: int = 6
    n_kv_head: int | None = None  # None -> n_head (standard multi-head attention)
    d_model: int = 384
    pos_emb: str = "learned"
    norm: str = "layernorm"
    mlp: str = "gelu"
    mlp_hidden: int | None = None  # None -> parameter-matched default, see __post_init__
    rope_theta: float = 10000.0
    dropout: float = 0.0
    tie_embeddings: bool = True

    def __post_init__(self):
        if self.n_kv_head is None:
            self.n_kv_head = self.n_head
        assert self.d_model % self.n_head == 0, "d_model must divide evenly into heads"
        assert self.n_head % self.n_kv_head == 0, "n_head must be a multiple of n_kv_head"
        assert self.pos_emb in ("learned", "rope")
        assert self.norm in ("layernorm", "rmsnorm")
        assert self.mlp in ("gelu", "swiglu")
        if self.pos_emb == "rope":
            assert (self.d_model // self.n_head) % 2 == 0, "RoPE needs an even head dimension"
        if self.mlp_hidden is None:
            if self.mlp == "gelu":
                self.mlp_hidden = 4 * self.d_model
            else:
                # SwiGLU has three weight matrices instead of two. A hidden size of
                # 8/3 * d gives 3 * d * (8/3 * d) = 8 * d^2 weights, the same as GELU's
                # 2 * d * 4d. Round up to a multiple of 64 for GPU-friendly shapes.
                hidden = int(8 * self.d_model / 3)
                self.mlp_hidden = 64 * math.ceil(hidden / 64)


class RMSNorm(nn.Module):
    """Scale by the root-mean-square of the features. Unlike LayerNorm there is no
    mean subtraction and no bias, which saves compute and (empirically) loses little."""

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x_f = x.float()
        x_f = x_f * torch.rsqrt(x_f.pow(2).mean(-1, keepdim=True) + self.eps)
        return x_f.type_as(x) * self.weight


def make_norm(cfg: GPTConfig) -> nn.Module:
    if cfg.norm == "rmsnorm":
        return RMSNorm(cfg.d_model)
    return nn.LayerNorm(cfg.d_model)


def rope_cache(head_dim: int, max_len: int, theta: float):
    """Precompute the rotation angles for rotary position embeddings.

    Feature pair i rotates at frequency theta^(-2i/head_dim): early pairs spin fast
    (fine position), late pairs spin slowly (coarse position)."""
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    angles = torch.outer(torch.arange(max_len).float(), inv_freq)  # (T, head_dim/2)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    """Rotate query/key features by their position. x: (B, H, T, head_dim).

    Splits the head into two halves and treats (x1[i], x2[i]) as a 2D point rotated
    by angle[t, i]. The dot product of a rotated query and key then depends only on
    their offset, which is what makes RoPE a relative position encoding."""
    x1, x2 = x.chunk(2, dim=-1)
    cos, sin = cos.to(x.dtype), sin.to(x.dtype)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class Attention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.n_head = cfg.n_head
        self.n_kv_head = cfg.n_kv_head
        self.head_dim = cfg.d_model // cfg.n_head
        self.dropout = cfg.dropout
        # Separate projections (not one fused qkv matrix) so grouped-query attention
        # can shrink k/v and LoRA can target each projection by name.
        self.q_proj = nn.Linear(cfg.d_model, self.n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.n_kv_head * self.head_dim, bias=False)
        self.o_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.resid_drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope=None):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        if rope is not None:
            cos, sin = rope
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)
        if self.n_kv_head != self.n_head:
            # Grouped-query attention: each k/v head is shared by a group of query heads.
            rep = self.n_head // self.n_kv_head
            k = k.repeat_interleave(rep, dim=1)
            v = v.repeat_interleave(rep, dim=1)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.o_proj(y))


class MLP(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.kind = cfg.mlp
        if cfg.mlp == "swiglu":
            self.gate = nn.Linear(cfg.d_model, cfg.mlp_hidden, bias=False)
        self.up = nn.Linear(cfg.d_model, cfg.mlp_hidden, bias=False)
        self.down = nn.Linear(cfg.mlp_hidden, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        if self.kind == "swiglu":
            # A learned, input-dependent gate decides how much of each hidden unit passes.
            h = F.silu(self.gate(x)) * self.up(x)
        else:
            h = F.gelu(self.up(x))
        return self.drop(self.down(h))


class Block(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.norm1 = make_norm(cfg)
        self.attn = Attention(cfg)
        self.norm2 = make_norm(cfg)
        self.mlp = MLP(cfg)

    def forward(self, x, rope=None):
        # Pre-norm: normalize the input to each sublayer, keep the residual path clean.
        x = x + self.attn(self.norm1(x), rope)
        x = x + self.mlp(self.norm2(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model) if cfg.pos_emb == "learned" else None
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm_f = make_norm(cfg)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight
        if cfg.pos_emb == "rope":
            cos, sin = rope_cache(cfg.d_model // cfg.n_head, cfg.block_size, cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        # GPT-2 scaling: each block adds two residual branches, so shrink the output
        # projections by 1/sqrt(2 * n_layer) to keep the residual stream's variance flat.
        for name, p in self.named_parameters():
            if name.endswith(("o_proj.weight", "down.weight")):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def num_params(self, non_embedding: bool = False) -> int:
        n = sum(p.numel() for p in self.parameters())  # tied weights are counted once
        if non_embedding:
            n -= self.tok_emb.weight.numel()
            if self.pos_emb is not None:
                n -= self.pos_emb.weight.numel()
        return n

    def forward(self, idx, targets=None):
        """idx: (B, T) token ids. targets: (B, T) next-token ids, -100 = ignore.
        Without targets, returns logits for the last position only (for sampling)."""
        B, T = idx.shape
        assert T <= self.cfg.block_size, f"sequence length {T} > block_size {self.cfg.block_size}"
        x = self.tok_emb(idx)
        if self.pos_emb is not None:
            x = x + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.drop(x)
        rope = (self.rope_cos[:T], self.rope_sin[:T]) if self.cfg.pos_emb == "rope" else None
        for block in self.blocks:
            x = block(x, rope)
        x = self.norm_f(x)
        if targets is None:
            return self.lm_head(x[:, [-1], :]), None
        logits = self.lm_head(x)
        loss = F.cross_entropy(
            logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100
        )
        return logits, loss

    def configure_optimizer(self, lr, weight_decay, betas, device_type):
        # Decay matrices only; norm gains are 1D and should not be pulled toward zero.
        params = [p for p in self.parameters() if p.requires_grad]
        decay = [p for p in params if p.dim() >= 2]
        no_decay = [p for p in params if p.dim() < 2]
        groups = [
            {"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0},
        ]
        return torch.optim.AdamW(groups, lr=lr, betas=betas, fused=device_type == "cuda")

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, stop_ids=()):
        """Sample tokens autoregressively. Stops early at a stop id (batch size 1 only)."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :].float()
            if temperature == 0:
                next_id = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k is not None:
                    kth = torch.topk(logits, min(top_k, logits.size(-1))).values[:, [-1]]
                    logits[logits < kth] = -float("inf")
                next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
            if stop_ids and idx.size(0) == 1 and next_id.item() in stop_ids:
                break
        return idx


In [ ]:
%%writefile slm/tokenizer.py
"""Byte-level BPE tokenizer trained on our own pretraining corpus.

The chat special tokens exist from the start, so instruction tuning never has to
resize the embedding matrix. Their ids are fixed by their order below.
"""

from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

SPECIAL_TOKENS = ["<|endoftext|>", "<|user|>", "<|assistant|>", "<|end|>"]
EOT_ID, USER_ID, ASSISTANT_ID, END_ID = range(len(SPECIAL_TOKENS))


def train_tokenizer(texts, vocab_size: int, out_path: str) -> Tokenizer:
    """Learn BPE merges from an iterable of strings and save to out_path (JSON).

    Byte-level means the base alphabet is the 256 byte values, so any input,
    including emoji or unseen scripts, encodes without an <unk> token."""
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    )
    tok.train_from_iterator(texts, trainer=trainer)
    for i, t in enumerate(SPECIAL_TOKENS):
        assert tok.token_to_id(t) == i, f"special token {t} got id {tok.token_to_id(t)}"
    tok.save(out_path)
    return tok


def load_tokenizer(path: str) -> Tokenizer:
    return Tokenizer.from_file(path)


In [ ]:
%%writefile slm/utils.py
"""Config loading, LR schedule, device setup and checkpoint helpers."""

import copy
import json
import math
import os
from contextlib import nullcontext

import torch

from .model import GPT, GPTConfig


def load_config(defaults: dict, path: str | None, overrides: list[str]) -> dict:
    """defaults <- JSON file <- key=value overrides. Dotted keys reach nested dicts,
    e.g. model.pos_emb=rope. Values parse as JSON when they can, else stay strings."""
    cfg = copy.deepcopy(defaults)
    if path:
        with open(path) as f:
            _merge(cfg, json.load(f))
    for item in overrides:
        key, _, raw = item.partition("=")
        try:
            value = json.loads(raw)
        except json.JSONDecodeError:
            value = raw
        node = cfg
        *parents, leaf = key.split(".")
        for k in parents:
            node = node.setdefault(k, {})
        node[leaf] = value
    return cfg


def _merge(base: dict, new: dict):
    for k, v in new.items():
        if isinstance(v, dict) and isinstance(base.get(k), dict):
            _merge(base[k], v)
        else:
            base[k] = v


def cosine_lr(it: int, max_iters: int, lr: float, min_lr: float, warmup_iters: int) -> float:
    """Linear warmup, then cosine decay from lr to min_lr."""
    if it < warmup_iters:
        return lr * (it + 1) / warmup_iters
    progress = min(1.0, (it - warmup_iters) / max(1, max_iters - warmup_iters))
    return min_lr + 0.5 * (1.0 + math.cos(math.pi * progress)) * (lr - min_lr)


def setup_device(requested: str = "cuda"):
    """Return (device, autocast context, grad scaler).

    Ampere and newer (RTX 30xx/40xx, A100, L4) train in bf16, which needs no loss
    scaling. Older GPUs such as Colab's T4 lack bf16, so they train in fp16 with a
    GradScaler that keeps small gradients from underflowing. On CPU, and in bf16,
    the scaler is disabled and its methods pass straight through."""
    if requested.startswith("cuda") and torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        if torch.cuda.get_device_capability() >= (8, 0):
            ctx = torch.autocast(device_type="cuda", dtype=torch.bfloat16)
            return requested, ctx, torch.amp.GradScaler("cuda", enabled=False)
        ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
        return requested, ctx, torch.amp.GradScaler("cuda", enabled=True)
    return "cpu", nullcontext(), torch.amp.GradScaler("cpu", enabled=False)


def save_checkpoint(path: str, model: GPT, **extra):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    state = {k.removeprefix("_orig_mod."): v for k, v in model.state_dict().items()}
    torch.save({"model": state, "model_config": vars(model.cfg).copy(), **extra}, path)


def load_checkpoint(path: str, device: str = "cpu"):
    """Return (model, checkpoint dict)."""
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = GPT(GPTConfig(**ckpt["model_config"]))
    model.load_state_dict(ckpt["model"])
    return model.to(device), ckpt


class JsonlLogger:
    def __init__(self, path: str):
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        self.f = open(path, "a", encoding="utf-8")

    def log(self, **row):
        self.f.write(json.dumps(row) + "\n")
        self.f.flush()


In [ ]:
%%writefile scripts/claims.py
"""Let several people's Colab sessions share one Drive folder without running the
same job twice.

A job is a run directory such as out/ablations/rope_s0. A worker claims it by
writing CLAIM there. The job counts as taken while any file in the directory
changed within STALE_MIN minutes: training appends to log.jsonl every few
seconds, so a live run keeps its claim fresh, while a run whose session died
goes stale and can be taken over (train.py then resumes from ckpt_last.pt).

    python scripts/claims.py --worker dhyan --out-dir out/sft/full_masked -- python scripts/sft.py ...

runs the command only if the job is neither finished nor running elsewhere.
"""

import argparse
import json
import os
import subprocess
import sys
import time

STALE_MIN = 15


def last_activity(out_dir: str) -> float:
    # The .claiming lock is not activity: it exists exactly while someone checks the job.
    times = [os.path.getmtime(os.path.join(out_dir, f)) for f in os.listdir(out_dir) if f != ".claiming"]
    return max(times, default=0.0)


def owner(out_dir: str) -> str | None:
    try:
        with open(os.path.join(out_dir, "CLAIM")) as f:
            return json.load(f)["worker"]
    except (OSError, ValueError, KeyError):
        return None


def status(out_dir: str, worker: str) -> str:
    """'done', 'failed', 'free', or 'running (<name>)'."""
    if os.path.exists(os.path.join(out_dir, "summary.json")):
        return "done"
    if os.path.exists(os.path.join(out_dir, "FAILED")):
        return "failed"
    if not os.path.isdir(out_dir):
        return "free"
    who = owner(out_dir)
    if who != worker and time.time() - last_activity(out_dir) < STALE_MIN * 60:
        return f"running ({who or 'someone'})"
    return "free"  # never started, abandoned, or our own run from a session that died


def claim(out_dir: str, worker: str):
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "CLAIM"), "w") as f:
        json.dump({"worker": worker, "time": time.strftime("%Y-%m-%d %H:%M:%S")}, f)


def try_claim(out_dir: str, worker: str) -> bool:
    """Claim out_dir if it is free, atomically: two workers checking the same job at
    the same instant cannot both get it. mkdir either creates the lock or fails."""
    os.makedirs(out_dir, exist_ok=True)
    lock = os.path.join(out_dir, ".claiming")
    try:
        os.mkdir(lock)
    except FileExistsError:
        try:
            if time.time() - os.path.getmtime(lock) < 60:
                return False  # another worker is claiming this job right now
            os.rmdir(lock)  # left behind by a worker that crashed mid-claim
        except FileNotFoundError:
            return False  # the other worker just finished claiming; let the caller look again
        return try_claim(out_dir, worker)
    try:
        if status(out_dir, worker) != "free":
            return False
        claim(out_dir, worker)
        return True
    finally:
        os.rmdir(lock)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--worker", required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("cmd", nargs=argparse.REMAINDER, help="-- command to run")
    args = ap.parse_args()
    cmd = args.cmd[1:] if args.cmd[:1] == ["--"] else args.cmd

    if not try_claim(args.out_dir, args.worker):
        print(f"skip {args.out_dir}: {status(args.out_dir, args.worker)}")
        return
    sys.exit(subprocess.run(cmd).returncode)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/eval_sft.py
"""Answer the held-out Dolly prompts with one or more checkpoints.

    python scripts/eval_sft.py base=out/final/ckpt.pt full_masked=out/sft/full_masked/ckpt.pt

Writes out/eval/<name>.jsonl (prompt, reference, response, stopped) and prints
the on-format rate: the share of answers that close with <|end|> before the
token limit. These files are the input to the pairwise LLM-judge comparison.
"""

import argparse
import json
import os
import sys

import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from slm.chat import prompt_ids  # noqa: E402
from slm.tokenizer import END_ID, load_tokenizer  # noqa: E402
from slm.utils import load_checkpoint, setup_device  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("models", nargs="+", help="name=path/to/ckpt.pt")
    ap.add_argument("--prompts", default="data/sft/eval_prompts.jsonl")
    ap.add_argument("--out-dir", default="out/eval")
    ap.add_argument("--max-new-tokens", type=int, default=256)
    ap.add_argument("--temperature", type=float, default=0.7)
    ap.add_argument("--top-k", type=int, default=50)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    with open(args.prompts, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]
    device, ctx, _ = setup_device("cuda")
    os.makedirs(args.out_dir, exist_ok=True)

    rates = {}
    for spec in args.models:
        name, path = spec.split("=", 1)
        model, ckpt = load_checkpoint(path, device)
        model.eval()
        tok = load_tokenizer(ckpt["tokenizer"])
        torch.manual_seed(args.seed)
        results, stopped = [], 0
        for r in rows:
            ids = prompt_ids(tok, r["prompt"])
            room = model.cfg.block_size - len(ids)
            if room <= 0:
                continue
            x = torch.tensor([ids], device=device)
            with ctx:
                out = model.generate(x, min(args.max_new_tokens, room), args.temperature, args.top_k, stop_ids=(END_ID,))
            new = out[0, len(ids) :].tolist()
            ended = bool(new) and new[-1] == END_ID
            stopped += ended
            results.append({**r, "response": tok.decode([t for t in new if t != END_ID]), "stopped": ended})
        with open(os.path.join(args.out_dir, f"{name}.jsonl"), "w", encoding="utf-8") as f:
            for res in results:
                f.write(json.dumps(res, ensure_ascii=False) + "\n")
        rates[name] = {"prompts": len(results), "on_format": stopped / len(results)}
        print(f"{name}: {len(results)} prompts, on-format (closed with <|end|>) {stopped / len(results):.1%}")
    with open(os.path.join(args.out_dir, "summary.json"), "w") as f:
        json.dump(rates, f, indent=2)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/generate.py
"""Sample from a checkpoint.

    python scripts/generate.py --ckpt out/ablations/modern_s0/ckpt.pt --prompt "Once upon a time"
    python scripts/generate.py --ckpt out/sft/full_masked/ckpt.pt --chat --prompt "Give three tips for sleeping well."
"""

import argparse
import os
import sys

import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from slm.chat import prompt_ids  # noqa: E402
from slm.tokenizer import END_ID, EOT_ID, load_tokenizer  # noqa: E402
from slm.utils import load_checkpoint, setup_device  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--prompt", default="")
    ap.add_argument("--chat", action="store_true", help="wrap the prompt in the chat template")
    ap.add_argument("--num-samples", type=int, default=1)
    ap.add_argument("--max-new-tokens", type=int, default=200)
    ap.add_argument("--temperature", type=float, default=0.8)
    ap.add_argument("--top-k", type=int, default=50)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    device, ctx, _ = setup_device("cuda")
    model, ckpt = load_checkpoint(args.ckpt, device)
    model.eval()
    tok = load_tokenizer(ckpt["tokenizer"])

    if args.chat:
        ids, stop = prompt_ids(tok, args.prompt), (END_ID,)
    else:
        ids, stop = [EOT_ID] + tok.encode(args.prompt).ids, (EOT_ID,)
    for _ in range(args.num_samples):
        x = torch.tensor([ids], device=device)
        with ctx:
            out = model.generate(x, args.max_new_tokens, args.temperature, args.top_k, stop_ids=stop)
        new = [t for t in out[0, len(ids) :].tolist() if t not in stop]
        print(args.prompt + ("\n" if args.chat else "") + tok.decode(new))
        print("-" * 60)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/pipeline.py
"""Run the whole project as a queue of jobs shared by several workers on one machine:
one CPU worker that prepares data, and one worker per GPU that trains.

    python scripts/pipeline.py prep --deadline <unix time>
    CUDA_VISIBLE_DEVICES=0 python scripts/pipeline.py gpu0 --deadline <unix time>
    CUDA_VISIBLE_DEVICES=1 python scripts/pipeline.py gpu1 --deadline <unix time>

GPU workers always take the most important job they can start: evaluation, then
fine-tuning, then the scaling sweep (largest model first, since it is the longest
job), then the ablations. Jobs are claimed through scripts/claims.py so no two
workers run the same one. Before the deadline, training pauses itself (resumable)
and no job is started that could not finish in time; a later session resumes.

Set SLM_SMOKE_TEST=1 to shrink every job to a few steps (for testing the plumbing).
"""

import argparse
import json
import os
import subprocess
import sys
import time
import traceback

from claims import status, try_claim
from sweep import SWEEPS

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
PY = sys.executable
PAUSED = 75
SFT_RUNS = ["full_masked", "full_unmasked", "lora_masked", "lora_unmasked"]
SMOKE = os.environ.get("SLM_SMOKE_TEST") == "1"

# name: (command, minutes needed). Data prep cannot pause, so it only starts with enough time left.
DATA = {
    "tinystories": ([PY, "scripts/prepare_pretrain.py", "--dataset", "tinystories"], 30),
    "sft": ([PY, "scripts/prepare_sft.py"], 10),
    "fineweb-edu": ([PY, "scripts/prepare_pretrain.py", "--dataset", "fineweb-edu", "--max-tokens", "1000000000"], 120),
}
SMOKE_DATA = {
    "tinystories": ["--max-tokens", "3000000", "--tokenizer-docs", "3000"],
    "fineweb-edu": ["--max-tokens", "3000000", "--val-docs", "300", "--tokenizer-docs", "3000"],
}
SMOKE_TIME = 30  # smoke-test jobs take seconds, so time requirements shrink by this factor
PREP_EXITED = os.path.join(ROOT, "data", ".prep_exited")


def need(minutes):
    """Minutes of session left that a step requires (shrunk in smoke tests)."""
    return minutes / (SMOKE_TIME if SMOKE else 1)
SMOKE_TRAIN = ["batch_size=4", "grad_accum=1", "max_iters=30", "eval_interval=10", "eval_iters=2", "log_interval=10"]
SMOKE_SFT = ["max_train_examples=200", "max_val_examples=32", "epochs=1", "batch_size=4", "grad_accum=1", "eval_interval=10"]


def log(worker, msg):
    print(f"[{worker} {time.strftime('%H:%M')}] {msg}", flush=True)


def data_state(name):
    d = os.path.join(ROOT, "data", name)
    if os.path.exists(os.path.join(d, ".complete")):
        return "ready"
    if os.path.exists(os.path.join(d, ".failed")):
        return "failed"
    # The prep worker has stopped (e.g. not enough time left), so this data won't arrive this session.
    return "unavailable" if os.path.exists(PREP_EXITED) else "pending"


def jobs(deadline):
    """Every job as a dict, most important first."""
    out = []
    ev = ["--max-new-tokens", "8"] if SMOKE else []
    for name in ["base"] + SFT_RUNS:
        ckpt = "out/scaling/l_s0/ckpt.pt" if name == "base" else f"out/sft/{name}/ckpt.pt"
        dep = "out/scaling/l_s0" if name == "base" else f"out/sft/{name}"
        out.append({"out_dir": f"out/eval/{name}", "deps": [dep], "data": ["sft"], "minutes": 45,
                    "cmd": [PY, "scripts/eval_sft.py", f"{name}={ckpt}", "--out-dir", f"out/eval/{name}", *ev]})
    for name in SFT_RUNS:
        mode, mask = name.split("_")
        out.append({"out_dir": f"out/sft/{name}", "deps": ["out/scaling/l_s0"], "data": ["sft"], "minutes": 40,
                    "cmd": [PY, "scripts/sft.py", "--config", f"configs/sft_{mode}.json", f"out_dir=out/sft/{name}",
                            f"mask_prompt={str(mask == 'masked').lower()}", *(SMOKE_SFT if SMOKE else [])]})
    for sweep, data, seeds in (("scaling", "fineweb-edu", [0]), ("ablations", "tinystories", [0, 1])):
        for name, (config, overrides) in SWEEPS[sweep].items():
            for seed in seeds:
                out_dir = f"out/{sweep}/{name}_s{seed}"
                # Training pauses itself at the deadline, so it may start with little time left.
                out.append({"out_dir": out_dir, "deps": [], "data": [data], "minutes": 5,
                            "cmd": [PY, "scripts/train.py", "--config", config, f"out_dir={out_dir}", f"seed={seed}",
                                    *overrides, f"deadline={deadline}", *(SMOKE_TRAIN if SMOKE else [])]})
    return out


def job_state(job, worker, known=None):
    """'done', 'failed', 'blocked', 'waiting', 'later' (not this session) or 'runnable'.
    known maps out_dir -> state for jobs already assessed, so 'later' and 'blocked'
    pass on to the jobs that depend on them."""
    known = known or {}
    st = status(os.path.join(ROOT, job["out_dir"]), worker)
    if st in ("done", "failed"):
        return st
    if st.startswith("running"):
        return "waiting"
    for dep in job["deps"]:
        dst = known.get(dep) or status(os.path.join(ROOT, dep), worker)
        if dst in ("failed", "blocked"):
            return "blocked"
        if dst == "later":
            return "later"
        if dst != "done":
            return "waiting"
    for name in job["data"]:
        state = data_state(name)
        if state == "failed":
            return "blocked"
        if state == "unavailable":
            return "later"
        if state != "ready":
            return "waiting"
    return "runnable"


def all_states(worker, deadline):
    """[(job, state)] in priority order. Assessed dependencies-first (the list runs
    from eval down to the ablations), so each job sees its dependencies' states."""
    all_jobs = jobs(deadline)
    known = {}
    for job in reversed(all_jobs):
        known[job["out_dir"]] = job_state(job, worker, known)
    return [(job, known[job["out_dir"]]) for job in all_jobs]


def run_job(job, worker):
    out_dir = os.path.join(ROOT, job["out_dir"])
    if not try_claim(out_dir, worker):
        return None  # another worker got it first
    log(worker, f"start {job['out_dir']}")
    t0 = time.time()
    with open(os.path.join(out_dir, "stdout.log"), "a", encoding="utf-8") as f:
        rc = subprocess.run(job["cmd"], cwd=ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
    minutes = (time.time() - t0) / 60
    if rc == 0:
        with open(os.path.join(out_dir, "summary.json")) as f:
            s = json.load(f)
        extra = f", val loss {s['final_val_loss']:.4f}" if "final_val_loss" in s else ""
        log(worker, f"done  {job['out_dir']} in {minutes:.0f} min{extra}")
    elif rc == PAUSED:
        log(worker, f"pause {job['out_dir']} at the deadline (resumable)")
    else:
        with open(os.path.join(out_dir, "stdout.log"), encoding="utf-8", errors="replace") as f:
            tail = f.readlines()[-15:]
        with open(os.path.join(out_dir, "FAILED"), "w") as f:
            f.write(f"exit code {rc}\n" + "".join(tail))
        log(worker, f"FAIL  {job['out_dir']} (exit {rc}); last lines:\n" + "".join(tail))
    return rc


def gpu_worker(worker, deadline):
    while True:
        left = (deadline - time.time()) / 60
        states = all_states(worker, deadline)
        runnable = [job for job, st in states if st == "runnable"]
        startable = [job for job in runnable if need(job["minutes"]) <= left]
        if startable:
            if run_job(startable[0], worker) == PAUSED:
                return "paused"
            continue
        if runnable:
            log(worker, f"{len(runnable)} job(s) left need more time than remains; stopping until the next session")
            return "paused"
        if any(st == "waiting" for _, st in states) and left > need(10):
            time.sleep(10 if SMOKE else 60)
            continue
        if all(st == "done" for _, st in states):
            log(worker, "all jobs done")
            return "complete"
        if any(st in ("later", "waiting") for _, st in states):
            log(worker, "the remaining jobs need data or time this session doesn't have; stopping until the next session")
            return "paused"
        stuck = [job["out_dir"] for job, st in states if st in ("failed", "blocked")]
        log(worker, f"nothing more I can do; failed or blocked: {', '.join(stuck) or 'none'}")
        return "stopped"


def prep_worker(deadline):
    for name, (cmd, minutes) in DATA.items():
        if data_state(name) == "ready":
            continue
        if (deadline - time.time()) / 60 < need(minutes):
            log("prep", f"not enough time left to prepare {name}; next session will")
            return "paused"
        d = os.path.join(ROOT, "data", name)
        os.makedirs(d, exist_ok=True)
        if os.path.exists(os.path.join(d, ".failed")):
            os.remove(os.path.join(d, ".failed"))
        full = cmd + (SMOKE_DATA.get(name, []) if SMOKE else [])
        log("prep", f"preparing {name}")
        t0 = time.time()
        with open(os.path.join(ROOT, "data", f"prepare_{name}.log"), "a", encoding="utf-8") as f:
            rc = subprocess.run(full, cwd=ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
        if rc == 0:
            open(os.path.join(d, ".complete"), "w").close()
            log("prep", f"{name} ready in {(time.time() - t0) / 60:.0f} min")
        else:
            open(os.path.join(d, ".failed"), "w").write(f"exit code {rc}\n")
            log("prep", f"FAIL preparing {name} (exit {rc}); see data/prepare_{name}.log")
    return "complete"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("worker", help="'prep' for data preparation, or a GPU worker name such as gpu0")
    ap.add_argument("--deadline", type=float, required=True, help="unix time by which to stop")
    args = ap.parse_args()
    os.chdir(ROOT)
    if args.worker == "prep":
        try:
            result = prep_worker(args.deadline)
        finally:
            open(PREP_EXITED, "w").close()
    else:
        result = None
        for attempt in range(5):
            try:
                result = gpu_worker(args.worker, args.deadline)
                break
            except Exception:
                # Losing a GPU worker for the rest of a long run costs hours; log and carry on.
                log(args.worker, "unexpected error (retrying in a minute):\n" + traceback.format_exc())
                time.sleep(5 if SMOKE else 60)
        else:
            result = "crashed"
    print(f"[{args.worker}] finished: {result}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/prepare_pretrain.py
"""Download a pretraining corpus, train a BPE tokenizer on it, and write token bins.

    python scripts/prepare_pretrain.py --dataset tinystories
    python scripts/prepare_pretrain.py --dataset fineweb-edu --max-tokens 1000000000

Writes <out-dir>/{train.bin, val.bin, tokenizer.json, meta.json}. Each document is
prefixed with <|endoftext|>, which doubles as the start token at fine-tuning time.
"""

import argparse
import faulthandler
import itertools
import json
import os
import sys
import tempfile
import time

import numpy as np
from tqdm import tqdm

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from slm.tokenizer import EOT_ID, load_tokenizer, train_tokenizer  # noqa: E402

DATASETS = {
    "tinystories": {"vocab_size": 8192},
    "fineweb-edu": {"vocab_size": 16384},
}
DOWNLOADS = os.path.join(tempfile.gettempdir(), "fineweb-edu")  # outside the output folder


def text_stream(dataset: str, split: str):
    """Yield documents one at a time without loading the corpus into memory."""
    from datasets import load_dataset

    if dataset == "tinystories":
        ds = load_dataset("roneneldan/TinyStories", split="train" if split == "train" else "validation")
        for row in ds:
            yield row["text"]
    else:
        # FineWeb-Edu has no validation split; the caller carves one off the front. Each ~2 GB
        # parquet file is downloaded whole, read in chunks, and deleted once read to the end
        # (a file only partly read is kept for the next pass). Streaming it with datasets
        # crashed (segfault) on Kaggle.
        import pyarrow.parquet as pq
        from huggingface_hub import hf_hub_download

        for i in range(14):
            path = hf_hub_download("HuggingFaceFW/fineweb-edu", f"sample/10BT/{i:03d}_00000.parquet",
                                   repo_type="dataset", local_dir=DOWNLOADS)
            pf = pq.ParquetFile(path)
            for batch in pf.iter_batches(batch_size=1024, columns=["text"]):
                yield from batch.column(0).to_pylist()
            pf.close()
            os.remove(path)


def beat(path: str | None, every_s: float = 60.0, _last=[0.0]):
    """Touch the heartbeat file (at most once a minute) so other sessions waiting
    for this data can tell the build is still alive."""
    if path and time.time() - _last[0] > every_s:
        with open(path, "w") as f:
            f.write(time.strftime("%Y-%m-%d %H:%M:%S"))
        _last[0] = time.time()


def write_bin(tok, texts, path: str, max_tokens: int | None, batch_docs: int = 1024, total_docs=None, heartbeat=None) -> tuple[int, int]:
    """Encode texts in batches and append uint16 ids to path. Returns (tokens, docs)."""
    n_tokens = n_docs = 0
    with open(path, "wb") as f, tqdm(total=total_docs, desc=os.path.basename(path), unit="doc") as bar:
        while True:
            batch = list(itertools.islice(texts, batch_docs))
            if not batch:
                break
            ids = []
            for enc in tok.encode_batch(batch):
                ids.append(EOT_ID)
                ids.extend(enc.ids)
            arr = np.array(ids, dtype=np.uint16)
            if max_tokens is not None and n_tokens + len(arr) > max_tokens:
                arr = arr[: max_tokens - n_tokens]
            arr.tofile(f)
            beat(heartbeat)
            n_tokens += len(arr)
            n_docs += len(batch)
            bar.update(len(batch))
            bar.set_postfix(tokens=f"{n_tokens / 1e6:.1f}M")
            if max_tokens is not None and n_tokens >= max_tokens:
                break
    return n_tokens, n_docs


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", choices=DATASETS, required=True)
    ap.add_argument("--out-dir", default=None, help="default: data/<dataset>")
    ap.add_argument("--vocab-size", type=int, default=None)
    ap.add_argument("--tokenizer-docs", type=int, default=200_000, help="documents used to learn BPE merges")
    ap.add_argument("--tokenizer", default=None, help="reuse an existing tokenizer.json instead of training one")
    ap.add_argument("--max-tokens", type=int, default=None, help="cap on training tokens (FineWeb-Edu)")
    ap.add_argument("--val-docs", type=int, default=5_000, help="FineWeb-Edu documents held out for validation")
    ap.add_argument("--heartbeat", default=None, help="file to touch every minute while working")
    args = ap.parse_args()
    faulthandler.enable()  # a crash in native code then still leaves a traceback in the log
    beat(args.heartbeat)

    out_dir = args.out_dir or os.path.join("data", args.dataset)
    os.makedirs(out_dir, exist_ok=True)
    vocab_size = args.vocab_size or DATASETS[args.dataset]["vocab_size"]
    tok_path = os.path.join(out_dir, "tokenizer.json").replace("\\", "/")  # portable across OSes
    fineweb = args.dataset == "fineweb-edu"
    # FineWeb-Edu docs 0..val_docs-1 are validation; the tokenizer learns from the docs after them.
    skip = args.val_docs if fineweb else 0

    if args.tokenizer:
        tok = load_tokenizer(args.tokenizer)
        tok.save(tok_path)
    elif os.path.exists(tok_path):
        print(f"reusing {tok_path}")
        tok = load_tokenizer(tok_path)
    else:
        print(f"training a {vocab_size}-token BPE tokenizer on {args.tokenizer_docs:,} documents")
        sample = itertools.islice(text_stream(args.dataset, "train"), skip, skip + args.tokenizer_docs)
        tok = train_tokenizer(sample, vocab_size, tok_path)
    assert tok.get_vocab_size() <= 65536, "uint16 bins need a vocabulary of at most 65,536"

    if fineweb:
        stream = text_stream(args.dataset, "train")
        val_tokens, val_docs = write_bin(tok, itertools.islice(stream, args.val_docs), os.path.join(out_dir, "val.bin"), None, total_docs=args.val_docs, heartbeat=args.heartbeat)
        train_tokens, train_docs = write_bin(tok, stream, os.path.join(out_dir, "train.bin"), args.max_tokens, heartbeat=args.heartbeat)
    else:
        val_tokens, val_docs = write_bin(tok, text_stream(args.dataset, "val"), os.path.join(out_dir, "val.bin"), None, heartbeat=args.heartbeat)
        train_tokens, train_docs = write_bin(tok, text_stream(args.dataset, "train"), os.path.join(out_dir, "train.bin"), args.max_tokens, heartbeat=args.heartbeat)

    meta = {
        "dataset": args.dataset,
        "vocab_size": tok.get_vocab_size(),
        "tokenizer": tok_path,
        "train_tokens": train_tokens,
        "train_docs": train_docs,
        "val_tokens": val_tokens,
        "val_docs": val_docs,
    }
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print(json.dumps(meta, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/prepare_sft.py
"""Download Alpaca-cleaned and Dolly 15k, normalize them to prompt/response pairs,
and hold out Dolly prompts for the instruction-following evaluation.

    python scripts/prepare_sft.py

Writes data/sft/{train.jsonl, val.jsonl, eval_prompts.jsonl, meta.json}.
"""

import argparse
import json
import os
import random


def join(instruction: str, extra: str) -> str:
    instruction, extra = instruction.strip(), (extra or "").strip()
    return f"{instruction}\n\n{extra}" if extra else instruction


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out-dir", default="data/sft")
    ap.add_argument("--n-eval", type=int, default=200, help="held-out Dolly prompts for evaluation")
    ap.add_argument("--max-eval-prompt-chars", type=int, default=1000, help="keep eval prompts short enough to leave room to answer")
    ap.add_argument("--val-frac", type=float, default=0.02)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    from datasets import load_dataset

    alpaca = [
        {"prompt": join(r["instruction"], r["input"]), "response": r["output"].strip(), "source": "alpaca"}
        for r in load_dataset("yahma/alpaca-cleaned", split="train")
    ]
    dolly = [
        {"prompt": join(r["instruction"], r["context"]), "response": r["response"].strip(), "source": "dolly", "category": r["category"]}
        for r in load_dataset("databricks/databricks-dolly-15k", split="train")
    ]
    alpaca = [r for r in alpaca if r["prompt"] and r["response"]]
    dolly = [r for r in dolly if r["prompt"] and r["response"]]

    rng = random.Random(args.seed)
    rng.shuffle(dolly)
    eval_rows, dolly_rest = [], []
    for r in dolly:
        if len(eval_rows) < args.n_eval and len(r["prompt"]) <= args.max_eval_prompt_chars:
            eval_rows.append({"prompt": r["prompt"], "reference": r["response"], "category": r["category"]})
        else:
            dolly_rest.append(r)

    pool = alpaca + dolly_rest
    rng.shuffle(pool)
    n_val = int(len(pool) * args.val_frac)
    val_rows, train_rows = pool[:n_val], pool[n_val:]

    os.makedirs(args.out_dir, exist_ok=True)
    write_jsonl(os.path.join(args.out_dir, "train.jsonl"), train_rows)
    write_jsonl(os.path.join(args.out_dir, "val.jsonl"), val_rows)
    write_jsonl(os.path.join(args.out_dir, "eval_prompts.jsonl"), eval_rows)
    meta = {"train": len(train_rows), "val": len(val_rows), "eval_prompts": len(eval_rows), "alpaca": len(alpaca), "dolly": len(dolly)}
    with open(os.path.join(args.out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print(json.dumps(meta, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/sft.py
"""Instruction-tune a pretrained checkpoint: full fine-tuning or LoRA, with the
loss on the response only (mask_prompt=true) or on every token.

    python scripts/sft.py --config configs/sft_full.json base_ckpt=out/final/ckpt.pt
    python scripts/sft.py --config configs/sft_lora.json mask_prompt=false out_dir=out/sft/lora_nomask

Validation loss is always measured on response tokens only, so masked and
unmasked runs are compared on the same quantity. LoRA weights are merged into
the saved checkpoint, so every SFT checkpoint loads like a pretrained one.
"""

import argparse
import json
import math
import os
import random
import sys
import time

import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from slm.chat import collate, encode_example  # noqa: E402
from slm.lora import apply_lora, lora_state_dict, merge_lora  # noqa: E402
from slm.tokenizer import load_tokenizer  # noqa: E402
from slm.utils import JsonlLogger, cosine_lr, load_checkpoint, load_config, save_checkpoint, setup_device  # noqa: E402

DEFAULTS = {
    "base_ckpt": "out/final/ckpt.pt",
    "sft_dir": "data/sft",
    "out_dir": "out/sft/run",
    "mode": "full",  # "full" | "lora"
    "mask_prompt": True,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_targets": ["q_proj", "k_proj", "v_proj", "o_proj", "gate", "up", "down"],
    "epochs": 2,
    "batch_size": 16,
    "grad_accum": 2,
    "lr": 1e-4,
    "min_lr_ratio": 0.1,
    "warmup_ratio": 0.03,
    "weight_decay": 0.0,
    "grad_clip": 1.0,
    "eval_interval": 200,
    "max_val_examples": 1000,
    "max_train_examples": None,
    "seed": 0,
    "device": "cuda",
}


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def encode_all(tok, rows, block_size, mask_prompt):
    out = [encode_example(tok, r["prompt"], r["response"], block_size, mask_prompt) for r in rows]
    kept = [e for e in out if e is not None]
    return kept, len(out) - len(kept)


@torch.no_grad()
def evaluate(model, examples, batch_size, device, ctx):
    model.eval()
    total, count = 0.0, 0
    for i in range(0, len(examples), batch_size):
        x, y = collate(examples[i : i + batch_size], device)
        with ctx:
            _, loss = model(x, y)
        n = (y != -100).sum().item()
        total += loss.item() * n
        count += n
    model.train()
    return total / count


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=None)
    ap.add_argument("overrides", nargs="*", help="key=value pairs")
    args = ap.parse_args()
    cfg = load_config(DEFAULTS, args.config, args.overrides)

    random.seed(cfg["seed"])
    torch.manual_seed(cfg["seed"])
    device, ctx, scaler = setup_device(cfg["device"])
    model, base = load_checkpoint(cfg["base_ckpt"], device)
    tok = load_tokenizer(base["tokenizer"])
    T = model.cfg.block_size

    train_rows = read_jsonl(os.path.join(cfg["sft_dir"], "train.jsonl"))
    if cfg["max_train_examples"]:
        train_rows = train_rows[: cfg["max_train_examples"]]
    val_rows = read_jsonl(os.path.join(cfg["sft_dir"], "val.jsonl"))[: cfg["max_val_examples"]]
    train, dropped = encode_all(tok, train_rows, T, cfg["mask_prompt"])
    val, _ = encode_all(tok, val_rows, T, mask_prompt=True)
    print(f"{len(train):,} train examples ({dropped:,} longer than {T} tokens dropped), {len(val):,} val")

    if cfg["mode"] == "lora":
        apply_lora(model, cfg["lora_r"], cfg["lora_alpha"], cfg["lora_dropout"], tuple(cfg["lora_targets"]))
        model.to(device)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"{cfg['mode']}: {trainable / 1e6:.2f}M of {total / 1e6:.2f}M parameters trainable")

    optimizer = model.configure_optimizer(cfg["lr"], cfg["weight_decay"], (0.9, 0.95), device.split(":")[0])
    examples_per_step = cfg["batch_size"] * cfg["grad_accum"]
    steps_per_epoch = math.ceil(len(train) / examples_per_step)
    max_steps = cfg["epochs"] * steps_per_epoch
    warmup = max(1, int(cfg["warmup_ratio"] * max_steps))
    os.makedirs(cfg["out_dir"], exist_ok=True)
    with open(os.path.join(cfg["out_dir"], "config.json"), "w") as f:
        json.dump(cfg, f, indent=2)
    logger = JsonlLogger(os.path.join(cfg["out_dir"], "log.jsonl"))
    if device.startswith("cuda"):
        torch.cuda.reset_peak_memory_stats()

    val_loss = evaluate(model, val, cfg["batch_size"], device, ctx)
    logger.log(step=0, val_loss=val_loss)
    print(f"step 0: val loss {val_loss:.4f}")
    t_start = time.time()
    step = 0
    model.train()
    for epoch in range(cfg["epochs"]):
        order = list(range(len(train)))
        random.shuffle(order)
        for s in range(steps_per_epoch):
            idx = order[s * examples_per_step : (s + 1) * examples_per_step]
            micro = [idx[i : i + cfg["batch_size"]] for i in range(0, len(idx), cfg["batch_size"])]
            lr = cosine_lr(step, max_steps, cfg["lr"], cfg["lr"] * cfg["min_lr_ratio"], warmup)
            for group in optimizer.param_groups:
                group["lr"] = lr
            for m in micro:
                x, y = collate([train[i] for i in m], device)
                with ctx:
                    _, loss = model(x, y)
                scaler.scale(loss / len(micro)).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            step += 1
            if step % 20 == 0:
                logger.log(step=step, epoch=epoch, train_loss=loss.item(), lr=lr)
                print(f"step {step}/{max_steps}: loss {loss.item():.4f}, lr {lr:.2e}")
            if step % cfg["eval_interval"] == 0 or step == max_steps:
                val_loss = evaluate(model, val, cfg["batch_size"], device, ctx)
                logger.log(step=step, val_loss=val_loss, elapsed=time.time() - t_start)
                print(f"step {step}: val loss {val_loss:.4f}")

    elapsed = time.time() - t_start
    peak_mem = torch.cuda.max_memory_allocated() / 1e9 if device.startswith("cuda") else None
    if cfg["mode"] == "lora":
        torch.save(lora_state_dict(model), os.path.join(cfg["out_dir"], "adapter.pt"))
        merge_lora(model)
    save_checkpoint(os.path.join(cfg["out_dir"], "ckpt.pt"), model, tokenizer=base["tokenizer"], sft_config=cfg, base_ckpt=cfg["base_ckpt"])
    summary = {
        "out_dir": cfg["out_dir"],
        "mode": cfg["mode"],
        "mask_prompt": cfg["mask_prompt"],
        "trainable_params": trainable,
        "total_params": total,
        "train_examples": len(train),
        "steps": max_steps,
        "final_val_loss": val_loss,
        "wall_time_s": elapsed,
        "gpu_hours": elapsed / 3600,
        "peak_mem_gb": peak_mem,
    }
    with open(os.path.join(cfg["out_dir"], "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
    print(f"done in {elapsed / 60:.1f} min, val loss {val_loss:.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/summarize.py
"""Tabulate and plot a sweep's results.

    python scripts/summarize.py out/ablations

Groups runs by variant (the folder name before _s<seed>), averages over seeds,
and writes <sweep_dir>/summary.md and val_curves.png. If the variants differ in
size by more than 2x, also plots final loss against parameters (scaling.png).
"""

import argparse
import glob
import json
import os
import statistics
from collections import defaultdict

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402


def load_runs(sweep_dir):
    runs = defaultdict(list)
    for path in sorted(glob.glob(os.path.join(sweep_dir, "*", "summary.json"))):
        run_dir = os.path.dirname(path)
        variant = os.path.basename(run_dir).rsplit("_s", 1)[0]
        with open(path) as f:
            summary = json.load(f)
        with open(os.path.join(run_dir, "log.jsonl")) as f:
            # A resumed run re-evaluates at its restart iteration; keep the last row per iteration.
            by_iter = {r["iter"]: r for r in map(json.loads, f) if "val_loss" in r}
        curve = [by_iter[i] for i in sorted(by_iter)]
        runs[variant].append({**summary, "curve": curve})
    return runs


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("sweep_dir")
    args = ap.parse_args()
    runs = load_runs(args.sweep_dir)
    if not runs:
        raise SystemExit(f"no finished runs under {args.sweep_dir}")

    rows = []
    for variant, rs in runs.items():
        losses = [r["final_val_loss"] for r in rs]
        rows.append({
            "variant": variant,
            "seeds": len(rs),
            "params_m": rs[0]["params"] / 1e6,
            "tokens_m": rs[0]["tokens"] / 1e6,
            "val_loss": statistics.mean(losses),
            "spread": (max(losses) - min(losses)) / 2,
            "tok_s": statistics.mean(r["tok_per_s"] for r in rs),
            "minutes": statistics.mean(r["wall_time_s"] for r in rs) / 60,
        })
    rows.sort(key=lambda r: r["val_loss"])
    base = next((r["val_loss"] for r in rows if r["variant"] == "baseline"), None)

    lines = [
        "| Variant | Seeds | Params (M) | Tokens (M) | Val loss | ± half-range | Δ vs baseline | Tok/s | Minutes |",
        "| --- | --- | --- | --- | --- | --- | --- | --- | --- |",
    ]
    for r in rows:
        delta = f"{r['val_loss'] - base:+.4f}" if base is not None else "–"
        lines.append(
            f"| {r['variant']} | {r['seeds']} | {r['params_m']:.2f} | {r['tokens_m']:.0f} | {r['val_loss']:.4f} | "
            f"{r['spread']:.4f} | {delta} | {r['tok_s']:,.0f} | {r['minutes']:.1f} |"
        )
    table = "\n".join(lines)
    with open(os.path.join(args.sweep_dir, "summary.md"), "w", encoding="utf-8") as f:
        f.write(table + "\n")
    print(table)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for variant, rs in sorted(runs.items()):
        curve = rs[0]["curve"]
        xs = [p["tokens"] / 1e6 for p in curve]
        ys = [statistics.mean(r["curve"][i]["val_loss"] for r in rs if i < len(r["curve"])) for i in range(len(curve))]
        ax.plot(xs[1:], ys[1:], label=variant)  # skip the step-0 loss, which flattens the plot
    ax.set_xlabel("Training tokens (millions)")
    ax.set_ylabel("Validation loss (nats/token)")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(args.sweep_dir, "val_curves.png"), dpi=150)

    sizes = [r["params_m"] for r in rows]
    if max(sizes) > 2 * min(sizes):
        fig, ax = plt.subplots(figsize=(6, 4.5))
        pts = sorted((r["params_m"], r["val_loss"], r["variant"]) for r in rows)
        ax.plot([p[0] for p in pts], [p[1] for p in pts], marker="o")
        for p, loss, name in pts:
            ax.annotate(name, (p, loss), textcoords="offset points", xytext=(6, 6))
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Parameters (millions, log scale)")
        ax.set_ylabel("Final validation loss (log scale)")
        ax.grid(alpha=0.3, which="both")
        fig.tight_layout()
        fig.savefig(os.path.join(args.sweep_dir, "scaling.png"), dpi=150)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/sweep.py
"""Run the architecture ablations or the scaling sweep, one training run at a time.

    python scripts/sweep.py ablations --seeds 0 1
    python scripts/sweep.py scaling --worker dhyan
    python scripts/sweep.py ablations --dry-run

Finished runs (those with a summary.json) are skipped, and so are runs another
worker is training right now (see scripts/claims.py), so several Colab sessions
can share one results folder. An interrupted run resumes from its checkpoint.
Summarize with scripts/summarize.py out/<sweep>.
"""

import argparse
import os
import random
import subprocess
import sys

from claims import status, try_claim

MODERN = ["model.pos_emb=rope", "model.norm=rmsnorm", "model.mlp=swiglu", "model.n_kv_head=2"]

SWEEPS = {
    # GPT-2-style baseline, then one LLaMA-style component swapped in at a time,
    # then all four together. Same data, token budget and eval batches for all.
    "ablations": {
        "baseline": ("configs/ablation_base.json", []),
        "rope": ("configs/ablation_base.json", ["model.pos_emb=rope"]),
        "rmsnorm": ("configs/ablation_base.json", ["model.norm=rmsnorm"]),
        "swiglu": ("configs/ablation_base.json", ["model.mlp=swiglu"]),
        "gqa": ("configs/ablation_base.json", ["model.n_kv_head=2"]),
        "modern": ("configs/ablation_base.json", MODERN),
    },
    # Three sizes trained on 20 tokens per parameter each, largest (slowest) first.
    "scaling": {
        "l": ("configs/scaling_l.json", []),
        "m": ("configs/scaling_m.json", []),
        "s": ("configs/scaling_s.json", []),
    },
}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("sweep", choices=SWEEPS)
    ap.add_argument("--seeds", type=int, nargs="+", default=[0])
    ap.add_argument("--only", nargs="*", help="run only these variants")
    ap.add_argument("--worker", default="local", help="your name, when sharing a results folder")
    ap.add_argument("--dry-run", action="store_true")
    ap.add_argument("extra", nargs="*", help="key=value overrides applied to every run")
    args = ap.parse_args()

    root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    jobs = [
        (name, seed, config, overrides)
        for name, (config, overrides) in SWEEPS[args.sweep].items()
        if not args.only or name in args.only
        for seed in args.seeds
    ]
    if args.sweep == "ablations":
        # Each worker takes the runs in its own order, so sessions started at the
        # same time rarely reach for the same run.
        random.Random(args.worker).shuffle(jobs)

    busy = []
    for name, seed, config, overrides in jobs:
        out_dir = f"out/{args.sweep}/{name}_s{seed}"
        st = status(os.path.join(root, out_dir), args.worker)
        if st != "free":
            print(f"skip {out_dir}: {st}")
            if st.startswith("running"):
                busy.append(out_dir)
            continue
        cmd = [sys.executable, "scripts/train.py", "--config", config, f"out_dir={out_dir}", f"seed={seed}", *overrides, *args.extra]
        print(" ".join(cmd), flush=True)
        if not args.dry_run:
            if not try_claim(os.path.join(root, out_dir), args.worker):
                print(f"skip {out_dir}: claimed by another worker just now")
                continue
            subprocess.run(cmd, cwd=root, check=True)
    if busy:
        print(f"still running in other sessions: {', '.join(busy)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/train.py
"""Pretrain a model from scratch on token bins.

    python scripts/train.py --config configs/ablation_base.json out_dir=out/test model.pos_emb=rope

Any key=value after the flags overrides the config (dotted keys reach model.*).
Writes <out_dir>/{log.jsonl, summary.json, ckpt.pt}.
"""

import argparse
import hashlib
import json
import math
import os
import sys
import time

import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from slm.data import TokenBins  # noqa: E402
from slm.model import GPT, GPTConfig  # noqa: E402
from slm.utils import JsonlLogger, cosine_lr, load_config, save_checkpoint, setup_device  # noqa: E402

DEFAULTS = {
    "data_dir": "data/tinystories",
    "out_dir": "out/run",
    "model": {},  # GPTConfig fields; vocab_size comes from the data's meta.json
    "batch_size": 32,
    "grad_accum": 4,
    "max_tokens": 100_000_000,  # training budget; ignored if max_iters or tokens_per_param is set
    "tokens_per_param": None,  # e.g. 20 -> budget = 20 x total parameters
    "max_iters": None,
    "lr": 1e-3,
    "min_lr": 1e-4,
    "warmup_iters": 100,
    "weight_decay": 0.1,
    "beta1": 0.9,
    "beta2": 0.95,
    "grad_clip": 1.0,
    "eval_interval": 250,
    "eval_iters": 40,
    "log_interval": 20,
    "seed": 0,
    "device": "cuda",
    "compile": False,  # torch.compile needs Triton, which is not bundled on Windows
    "save_checkpoint": True,
    "resume": True,  # continue from <out_dir>/ckpt_last.pt if a previous session was cut off
    "deadline": None,  # unix time; save resume state and exit with PAUSED once it passes
}

PAUSED = 75  # exit code: stopped at the deadline, resumable


def dataset_id(meta: dict) -> str:
    """Fingerprint of the token data: the tokenizer file plus the token counts."""
    with open(meta["tokenizer"], "rb") as f:
        digest = hashlib.sha1(f.read()).hexdigest()
    return f"{digest}:{meta['train_tokens']}:{meta['val_tokens']}"


def save_resume_state(path, model, optimizer, scaler, it, best_val, elapsed, data_id):
    """Everything needed to continue mid-run. Written to a temp file first, so a
    disconnect during the write cannot corrupt the previous state."""
    state = {
        "model": {k.removeprefix("_orig_mod."): v for k, v in model.state_dict().items()},
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "iter": it,
        "best_val": best_val,
        "elapsed": elapsed,
        "rng": torch.get_rng_state(),
        "data_id": data_id,
    }
    torch.save(state, path + ".tmp")
    os.replace(path + ".tmp", path)


@torch.no_grad()
def evaluate(model, batches, device, ctx):
    model.eval()
    losses = []
    for x, y in batches:
        with ctx:
            _, loss = model(x.to(device), y.to(device))
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=None)
    ap.add_argument("overrides", nargs="*", help="key=value pairs")
    args = ap.parse_args()
    cfg = load_config(DEFAULTS, args.config, args.overrides)

    torch.manual_seed(cfg["seed"])
    device, ctx, scaler = setup_device(cfg["device"])

    with open(os.path.join(cfg["data_dir"], "meta.json")) as f:
        meta = json.load(f)
    model_cfg = GPTConfig(**{**cfg["model"], "vocab_size": meta["vocab_size"]})
    model = GPT(model_cfg).to(device)
    n_params = model.num_params()
    n_params_nonemb = model.num_params(non_embedding=True)

    T = model_cfg.block_size
    tokens_per_iter = cfg["batch_size"] * cfg["grad_accum"] * T
    if cfg["max_iters"] is not None:
        max_iters = cfg["max_iters"]
    else:
        budget = cfg["tokens_per_param"] * n_params if cfg["tokens_per_param"] else cfg["max_tokens"]
        max_iters = math.ceil(budget / tokens_per_iter)

    os.makedirs(cfg["out_dir"], exist_ok=True)
    with open(os.path.join(cfg["out_dir"], "config.json"), "w") as f:
        json.dump({**cfg, "model": vars(model_cfg)}, f, indent=2)
    print(f"model: {n_params / 1e6:.2f}M params ({n_params_nonemb / 1e6:.2f}M non-embedding)")
    print(f"training: {max_iters} iters x {tokens_per_iter:,} tokens = {max_iters * tokens_per_iter / 1e6:.0f}M tokens on {device}")

    data = TokenBins(cfg["data_dir"], T, device)
    val_batches = data.fixed_batches("val", cfg["eval_iters"], cfg["batch_size"])
    optimizer = model.configure_optimizer(cfg["lr"], cfg["weight_decay"], (cfg["beta1"], cfg["beta2"]), device.split(":")[0])
    train_model = torch.compile(model) if cfg["compile"] else model
    if device.startswith("cuda"):
        torch.cuda.reset_peak_memory_stats()

    last_path = os.path.join(cfg["out_dir"], "ckpt_last.pt")
    log_path = os.path.join(cfg["out_dir"], "log.jsonl")
    data_id = dataset_id(meta)
    start_it, best_val, elapsed_before = 0, float("inf"), 0.0
    if cfg["resume"] and os.path.exists(last_path):
        state = torch.load(last_path, map_location=device, weights_only=False)
        if state.get("data_id") == data_id:
            model.load_state_dict(state["model"])
            optimizer.load_state_dict(state["optimizer"])
            scaler.load_state_dict(state["scaler"])
            torch.set_rng_state(state["rng"].cpu())  # the CPU generator needs a CPU tensor
            start_it, best_val, elapsed_before = state["iter"], state["best_val"], state["elapsed"]
            print(f"resumed from iter {start_it}")
        else:
            # The token data was rebuilt since this state was saved (e.g. in a new Colab
            # session), so its token ids may not match the model's. Start over.
            print("resume state was saved with different data; starting this run fresh")
    if start_it == 0 and os.path.exists(log_path):
        os.remove(log_path)
    logger = JsonlLogger(log_path)

    x, y = data.get_batch("train", cfg["batch_size"])
    t_start = t_last = time.time()
    for it in range(start_it, max_iters + 1):
        lr = cosine_lr(it, max_iters, cfg["lr"], cfg["min_lr"], cfg["warmup_iters"])
        for group in optimizer.param_groups:
            group["lr"] = lr

        if it % cfg["eval_interval"] == 0 or it == max_iters:
            val_loss = evaluate(model, val_batches, device, ctx)
            elapsed = elapsed_before + time.time() - t_start
            logger.log(iter=it, tokens=it * tokens_per_iter, val_loss=val_loss, elapsed=elapsed)
            print(f"iter {it}: val loss {val_loss:.4f}")
            if val_loss < best_val:
                best_val = val_loss
                if cfg["save_checkpoint"] and it > 0:
                    save_checkpoint(os.path.join(cfg["out_dir"], "ckpt.pt"), model, iter=it, val_loss=val_loss, tokenizer=meta["tokenizer"], train_config=cfg)
            if it > start_it and it < max_iters:
                save_resume_state(last_path, model, optimizer, scaler, it, best_val, elapsed, data_id)
        if it == max_iters:
            break

        for _ in range(cfg["grad_accum"]):
            with ctx:
                _, loss = train_model(x, y)
            x, y = data.get_batch("train", cfg["batch_size"])  # fetch the next batch while the GPU works
            scaler.scale(loss / cfg["grad_accum"]).backward()
        scaler.unscale_(optimizer)  # so clipping sees true gradient sizes under fp16
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"]).item()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        if cfg["deadline"] and time.time() > cfg["deadline"]:
            # Out of time (e.g. a 12-hour Kaggle run): keep everything for the next session.
            elapsed = elapsed_before + time.time() - t_start
            save_resume_state(last_path, model, optimizer, scaler, it + 1, best_val, elapsed, data_id)
            print(f"deadline reached after iter {it}; paused, resumable from {last_path}")
            sys.exit(PAUSED)

        if it % cfg["log_interval"] == 0:
            now = time.time()
            tok_s = tokens_per_iter * (cfg["log_interval"] if it > start_it else 1) / (now - t_last)
            t_last = now
            logger.log(iter=it, tokens=(it + 1) * tokens_per_iter, train_loss=loss.item(), lr=lr, grad_norm=grad_norm, tok_per_s=tok_s)
            print(f"iter {it}: loss {loss.item():.4f}, lr {lr:.2e}, grad norm {grad_norm:.2f}, {tok_s:,.0f} tok/s")

    elapsed = elapsed_before + time.time() - t_start
    if os.path.exists(last_path):
        os.remove(last_path)  # the run is complete; ckpt.pt holds the best weights
    summary = {
        "out_dir": cfg["out_dir"],
        "model": vars(model_cfg),
        "seed": cfg["seed"],
        "params": n_params,
        "params_non_embedding": n_params_nonemb,
        "tokens": max_iters * tokens_per_iter,
        "final_val_loss": val_loss,
        "best_val_loss": best_val,
        "wall_time_s": elapsed,
        "tok_per_s": max_iters * tokens_per_iter / elapsed,
        "peak_mem_gb": torch.cuda.max_memory_allocated() / 1e9 if device.startswith("cuda") else None,
    }
    with open(os.path.join(cfg["out_dir"], "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
    print(f"done in {elapsed / 60:.1f} min, final val loss {val_loss:.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile configs/ablation_base.json
{
  "data_dir": "data/tinystories",
  "model": {
    "block_size": 512,
    "n_layer": 6,
    "n_head": 6,
    "d_model": 384,
    "pos_emb": "learned",
    "norm": "layernorm",
    "mlp": "gelu"
  },
  "batch_size": 32,
  "grad_accum": 4,
  "max_tokens": 100000000,
  "lr": 1e-3,
  "min_lr": 1e-4,
  "warmup_iters": 100,
  "eval_interval": 100,
  "eval_iters": 40
}


In [ ]:
%%writefile configs/scaling_l.json
{
  "data_dir": "data/fineweb-edu",
  "model": {
    "block_size": 1024,
    "n_layer": 12,
    "n_head": 8,
    "n_kv_head": 4,
    "d_model": 512,
    "pos_emb": "rope",
    "norm": "rmsnorm",
    "mlp": "swiglu"
  },
  "batch_size": 16,
  "grad_accum": 8,
  "tokens_per_param": 20,
  "lr": 1e-3,
  "min_lr": 1e-4,
  "warmup_iters": 200,
  "eval_interval": 250,
  "eval_iters": 40
}


In [ ]:
%%writefile configs/scaling_m.json
{
  "data_dir": "data/fineweb-edu",
  "model": {
    "block_size": 1024,
    "n_layer": 8,
    "n_head": 6,
    "n_kv_head": 3,
    "d_model": 384,
    "pos_emb": "rope",
    "norm": "rmsnorm",
    "mlp": "swiglu"
  },
  "batch_size": 16,
  "grad_accum": 8,
  "tokens_per_param": 20,
  "lr": 1e-3,
  "min_lr": 1e-4,
  "warmup_iters": 200,
  "eval_interval": 250,
  "eval_iters": 40
}


In [ ]:
%%writefile configs/scaling_s.json
{
  "data_dir": "data/fineweb-edu",
  "model": {
    "block_size": 1024,
    "n_layer": 6,
    "n_head": 4,
    "n_kv_head": 2,
    "d_model": 256,
    "pos_emb": "rope",
    "norm": "rmsnorm",
    "mlp": "swiglu"
  },
  "batch_size": 16,
  "grad_accum": 8,
  "tokens_per_param": 20,
  "lr": 1e-3,
  "min_lr": 1e-4,
  "warmup_iters": 200,
  "eval_interval": 250,
  "eval_iters": 40
}


In [ ]:
%%writefile configs/sft_full.json
{
  "base_ckpt": "out/scaling/l_s0/ckpt.pt",
  "out_dir": "out/sft/full_masked",
  "mode": "full",
  "mask_prompt": true,
  "epochs": 2,
  "batch_size": 16,
  "grad_accum": 2,
  "lr": 1e-4
}


In [ ]:
%%writefile configs/sft_lora.json
{
  "base_ckpt": "out/scaling/l_s0/ckpt.pt",
  "out_dir": "out/sft/lora_masked",
  "mode": "lora",
  "mask_prompt": true,
  "lora_r": 16,
  "lora_alpha": 32,
  "epochs": 2,
  "batch_size": 16,
  "grad_accum": 2,
  "lr": 1e-3
}


In [ ]:
%%writefile tests/test_claims.py
import json
import os
import sys
import threading
import time

sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "scripts"))
from claims import STALE_MIN, status, try_claim  # noqa: E402


def test_only_one_of_many_simultaneous_claims_wins(tmp_path):
    out = str(tmp_path / "run")
    results = []
    barrier = threading.Barrier(8)

    def worker(i):
        barrier.wait()
        results.append(try_claim(out, f"w{i}"))

    threads = [threading.Thread(target=worker, args=(i,)) for i in range(8)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    assert results.count(True) == 1
    assert not os.path.exists(os.path.join(out, ".claiming"))


def test_status_lifecycle(tmp_path):
    out = str(tmp_path / "run")
    assert status(out, "a") == "free"
    assert try_claim(out, "a")
    assert status(out, "b") == "running (a)"
    assert status(out, "a") == "free"  # our own claim: a restarted session may resume it
    assert not try_claim(out, "b")

    old = time.time() - (STALE_MIN + 1) * 60  # the session running it died long ago
    for f in os.listdir(out):
        os.utime(os.path.join(out, f), (old, old))
    assert status(out, "b") == "free"
    assert try_claim(out, "b")
    assert json.load(open(os.path.join(out, "CLAIM")))["worker"] == "b"

    open(os.path.join(out, "FAILED"), "w").close()
    assert status(out, "a") == "failed"
    open(os.path.join(out, "summary.json"), "w").close()
    assert status(out, "a") == "done"  # a finished run wins over an earlier failure


def test_lock_released_between_checks_is_not_a_crash(tmp_path, monkeypatch):
    """Another worker holds the lock, then releases it before we can read its age."""
    import claims

    out = str(tmp_path / "run")
    os.makedirs(os.path.join(out, ".claiming"))

    def vanished(path):
        raise FileNotFoundError(path)

    monkeypatch.setattr(claims.os.path, "getmtime", vanished)
    assert claims.try_claim(out, "a") is False


In [ ]:
%%writefile tests/test_model.py
import itertools
import math
import os
import sys

import pytest
import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from slm.chat import IGNORE, collate, encode_example, prompt_ids  # noqa: E402
from slm.lora import LoRALinear, apply_lora, merge_lora  # noqa: E402
from slm.model import GPT, GPTConfig, apply_rope, rope_cache  # noqa: E402
from slm.utils import cosine_lr, load_config  # noqa: E402

VARIANTS = [
    dict(pos_emb=p, norm=n, mlp=m, n_kv_head=kv)
    for p, n, m, kv in itertools.product(["learned", "rope"], ["layernorm", "rmsnorm"], ["gelu", "swiglu"], [4, 2])
]


def tiny(**kw):
    base = dict(vocab_size=97, block_size=32, n_layer=2, n_head=4, d_model=32)
    return GPTConfig(**{**base, **kw})


@pytest.mark.parametrize("variant", VARIANTS, ids=lambda v: "-".join(str(x) for x in v.values()))
def test_forward_shapes_and_initial_loss(variant):
    torch.manual_seed(0)
    model = GPT(tiny(**variant))
    x = torch.randint(0, 97, (3, 32))
    y = torch.randint(0, 97, (3, 32))
    logits, loss = model(x, y)
    assert logits.shape == (3, 32, 97)
    # A fresh model should be close to uniform over the vocabulary. (Targets must
    # differ from the inputs: tied embeddings make a fresh model favour echoing
    # its input token, so loss on targets == inputs starts well below log(V).)
    assert abs(loss.item() - math.log(97)) < 0.5


@pytest.mark.parametrize("variant", VARIANTS, ids=lambda v: "-".join(str(x) for x in v.values()))
def test_causal(variant):
    """Changing a future token must not change predictions at earlier positions."""
    torch.manual_seed(0)
    model = GPT(tiny(**variant)).eval()
    x = torch.randint(0, 97, (1, 32))
    x2 = x.clone()
    x2[0, 20] = (x2[0, 20] + 1) % 97
    y = torch.zeros_like(x)
    l1, _ = model(x, y)
    l2, _ = model(x2, y)
    assert torch.allclose(l1[:, :20], l2[:, :20], atol=1e-5)
    assert not torch.allclose(l1[:, 20:], l2[:, 20:], atol=1e-5)


def test_rope_is_relative():
    """q.k after RoPE depends only on the offset between positions."""
    torch.manual_seed(0)
    cos, sin = rope_cache(16, 64, 10000.0)
    q, k = torch.randn(16), torch.randn(16)

    def score(i, j):
        qi = apply_rope(q.view(1, 1, 1, 16), cos[i : i + 1], sin[i : i + 1])
        kj = apply_rope(k.view(1, 1, 1, 16), cos[j : j + 1], sin[j : j + 1])
        return (qi * kj).sum().item()

    assert score(5, 2) == pytest.approx(score(40, 37), abs=1e-4)
    assert score(5, 2) != pytest.approx(score(5, 4), abs=1e-4)


def test_swiglu_matches_gelu_parameter_count():
    gelu = GPT(GPTConfig(d_model=384, n_head=6, mlp="gelu")).num_params()
    swiglu = GPT(GPTConfig(d_model=384, n_head=6, mlp="swiglu")).num_params()
    assert gelu == swiglu  # 8/3 * 384 = 1024 exactly, so no rounding slack


def test_gqa_shrinks_kv_projections():
    mha = GPT(tiny(n_kv_head=4))
    gqa = GPT(tiny(n_kv_head=2))
    assert gqa.blocks[0].attn.k_proj.weight.shape == (16, 32)
    assert mha.num_params() - gqa.num_params() == 2 * 2 * 32 * 16  # layers x (k, v) x d x saved width


def test_lora_starts_identical_and_merges_exactly():
    torch.manual_seed(0)
    model = GPT(tiny(mlp="swiglu")).eval()
    x = torch.randint(0, 97, (2, 16))
    before, _ = model(x, x)
    apply_lora(model, r=4, alpha=8, targets=("q_proj", "v_proj", "up"))
    after, _ = model(x, x)
    assert torch.allclose(before, after, atol=1e-6)  # B = 0 at init
    trainable = [n for n, p in model.named_parameters() if p.requires_grad]
    assert trainable and all("lora_" in n for n in trainable)

    for m in model.modules():  # pretend training moved B away from zero
        if isinstance(m, LoRALinear):
            torch.nn.init.normal_(m.lora_B, std=0.1)
    adapted, _ = model(x, x)
    merge_lora(model)
    merged, _ = model(x, x)
    assert not any(isinstance(m, LoRALinear) for m in model.modules())
    assert torch.allclose(adapted, merged, atol=1e-5)


class FakeTok:
    """Stand-in tokenizer: one id per character, offset past the special tokens."""

    class Enc:
        def __init__(self, ids):
            self.ids = ids

    def encode(self, s):
        return self.Enc([4 + (ord(c) % 90) for c in s])


def test_sft_masking_covers_only_the_response():
    tok = FakeTok()
    x, y = encode_example(tok, "hi", "yo", block_size=64, mask_prompt=True)
    p = prompt_ids(tok, "hi")
    assert len(x) == len(y)
    assert y[: len(p) - 1] == [IGNORE] * (len(p) - 1)
    assert y[len(p) - 1 :] == tok.encode("yo").ids + [3]  # response, then <|end|>
    _, y_all = encode_example(tok, "hi", "yo", block_size=64, mask_prompt=False)
    assert IGNORE not in y_all
    assert encode_example(tok, "x" * 100, "yo", block_size=64) is None

    xs, ys = collate([(x, y), (x[:3], y[:3])], "cpu")
    assert xs.shape == ys.shape == (2, len(x))
    assert (ys[1, 3:] == IGNORE).all()


def test_tiny_model_overfits_one_batch():
    torch.manual_seed(0)
    model = GPT(tiny(pos_emb="rope", norm="rmsnorm", mlp="swiglu", n_kv_head=2))
    opt = model.configure_optimizer(3e-3, 0.0, (0.9, 0.95), "cpu")
    x = torch.randint(0, 97, (4, 32))
    first = None
    for _ in range(150):
        _, loss = model(x[:, :-1], x[:, 1:])
        first = first or loss.item()
        opt.zero_grad()
        loss.backward()
        opt.step()
    assert loss.item() < 0.2 * first


def test_cosine_lr_schedule():
    assert cosine_lr(0, 1000, 1e-3, 1e-4, 100) == pytest.approx(1e-5)
    assert cosine_lr(99, 1000, 1e-3, 1e-4, 100) == pytest.approx(1e-3)
    assert cosine_lr(1000, 1000, 1e-3, 1e-4, 100) == pytest.approx(1e-4)


def test_config_overrides(tmp_path):
    path = tmp_path / "c.json"
    path.write_text('{"lr": 0.001, "model": {"n_layer": 6, "mlp": "gelu"}}')
    cfg = load_config({"lr": 1, "seed": 0, "model": {}}, str(path), ["model.mlp=swiglu", "seed=3", "out_dir=out/x"])
    assert cfg == {"lr": 0.001, "seed": 3, "model": {"n_layer": 6, "mlp": "swiglu"}, "out_dir": "out/x"}


In [ ]:
%%writefile tests/test_scripts_compile.py
import glob
import os
import py_compile

import pytest

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


@pytest.mark.parametrize("path", sorted(glob.glob(os.path.join(ROOT, "scripts", "*.py"))), ids=os.path.basename)
def test_script_compiles(path):
    """Scripts run as subprocesses on Kaggle/Colab, where a syntax error surfaces hours in."""
    py_compile.compile(path, doraise=True)


## Helpers

In [ ]:
from IPython.display import Image, display

def sh(cmd):
    """Run a shell command with live output; stop the notebook if it fails."""
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get('_exit_code', 0)
    if code:
        raise RuntimeError(f'exit code {code}: {cmd}')

def age_min(path):
    return (time.time() - os.path.getmtime(path)) / 60

def ensure_data(name, cmd):
    """Get data/<name> onto this machine: copy it from the shared Drive cache, wait while a teammate
    is building it, or build it here and cache it for everyone."""
    local, cache = f'data/{name}', f'{DRIVE}/data_cache/{name}'
    done = f'{cache}/.complete'   # written last, so a half-finished copy is never trusted
    building = f'{cache}/.building'  # heartbeat of whoever is building it
    if os.path.exists(f'{local}/meta.json') and os.path.exists(done):
        return
    while not os.path.exists(done) and os.path.exists(building) and age_min(building) < 20 \
            and (open(building).read().split() or [''])[0] != WORKER:
        print(f'a teammate is preparing {name}; checking again in a minute', flush=True)
        time.sleep(60)
    if os.path.exists(done):
        if not os.path.exists(f'{local}/meta.json'):
            print(f'copying {name} data from Drive', flush=True)
            shutil.copytree(cache, local, dirs_exist_ok=True)
        return
    os.makedirs(cache, exist_ok=True)
    beat = lambda: open(building, 'w').write(f'{WORKER} {time.ctime()}')
    beat()
    if not os.path.exists(f'{local}/meta.json'):
        sh(f'{cmd} --heartbeat {building}' if 'prepare_pretrain' in cmd else cmd)
    beat()
    print(f'caching {name} data to Drive for the team', flush=True)
    shutil.copytree(local, cache, dirs_exist_ok=True)
    open(done, 'w').close()
    os.remove(building)

def claimed(out_dir, cmd):
    """Run cmd unless out_dir is finished or a teammate is running it right now."""
    sh(f'python scripts/claims.py --worker {WORKER} --out-dir {out_dir} -- {cmd}')

def done(out_dir):
    return os.path.exists(f'{out_dir}/summary.json')

def show(path):
    if os.path.exists(path):
        display(Image(path))

## Check the code
Unit tests: causality, RoPE, LoRA, loss masking, overfitting a batch. About 30 seconds.

In [ ]:
sh('python -m pytest tests -q')

## Step 1 — Architecture ablations (extension 1)
GPT-2-style baseline vs. RoPE, RMSNorm, SwiGLU and grouped-query attention, swapped in one at a time, plus all four together. ~14M parameters, 100M TinyStories tokens per run, 2 seeds each: 12 runs of ~19 minutes on a T4.

In [ ]:
def step1():
    ensure_data('tinystories', 'python scripts/prepare_pretrain.py --dataset tinystories')
    sh(f'python scripts/sweep.py ablations --seeds 0 1 --worker {WORKER}')
    if os.path.isdir('out/ablations') and any(done(f'out/ablations/{d}') for d in os.listdir('out/ablations')):
        sh('python scripts/summarize.py out/ablations')
        show('out/ablations/val_curves.png')

## Step 2 — Scaling sweep (extension 2)
Three sizes (~9M, ~20M, ~46M parameters) on FineWeb-Edu, 20 tokens per parameter each. The 46M model is the longest job in the project (~7 h on a T4, spread over sessions). The first person to get here prepares 1B FineWeb-Edu tokens (~1 h); everyone else copies them from Drive.

In [ ]:
def step2():
    ensure_data('fineweb-edu', 'python scripts/prepare_pretrain.py --dataset fineweb-edu --max-tokens 1000000000')
    sh(f'python scripts/sweep.py scaling --worker {WORKER}')
    if os.path.isdir('out/scaling') and any(done(f'out/scaling/{d}') for d in os.listdir('out/scaling')):
        sh('python scripts/summarize.py out/scaling')
        show('out/scaling/val_curves.png')
        show('out/scaling/scaling.png')

## Step 3 — Instruction tuning (extension 3)
Fine-tunes the 46M model four ways: full vs. LoRA, loss on the response only vs. on every token. Starts once Step 2's 46M model is finished.

In [ ]:
SFT_RUNS = ['full_masked', 'full_unmasked', 'lora_masked', 'lora_unmasked']

def step3():
    if not done('out/scaling/l_s0'):
        print('Step 3 needs the finished 46M model (out/scaling/l_s0); skipping it for now.')
        return
    ensure_data('sft', 'python scripts/prepare_sft.py')
    for name in SFT_RUNS:
        mode, mask = name.split('_')
        claimed(f'out/sft/{name}', f'python scripts/sft.py --config configs/sft_{mode}.json '
                                   f'out_dir=out/sft/{name} mask_prompt={str(mask == "masked").lower()}')

## Step 4 — Answer the held-out prompts
The base model and the four tuned models answer the 200 held-out Dolly prompts; answers are saved to `out/eval/` for the pairwise comparison.

In [ ]:
def step4():
    if not all(done(f'out/sft/{n}') for n in SFT_RUNS):
        print('Step 4 needs all four fine-tuned models from Step 3; skipping it for now.')
        return
    models = ['base=out/scaling/l_s0/ckpt.pt'] + [f'{n}=out/sft/{n}/ckpt.pt' for n in SFT_RUNS]
    claimed('out/eval', 'python scripts/eval_sft.py ' + ' '.join(models))

## Run
Works through `STEPS` in order, skipping jobs that are finished or that a teammate is running.

In [ ]:
for s in STEPS:
    print(f'\n=== Step {s} ===', flush=True)
    {1: step1, 2: step2, 3: step3, 4: step4}[s]()
print('\nNothing left for this session. Jobs listed as running belong to teammates; Run all again later to pick up whatever is left.')